# Part 6：短時間 Marker-aligned Profiling Horizon

## 研究問題

只看 warmup 後前 K 個已知 iterations，短 profiling 是否足以預測？

## 目前結論

K=50 的 behavior_mean_duration combined MAPE 約 3.77%。短 prefix 有效，但 K 與 iteration boundaries 仍由 marker 提供。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 以 marker 取得 warmup 後 iteration boundaries。
2. 截取 K=5、10、20、50、108 的 prefix。
3. 建立 behavior、distribution 與 duration features。
4. 在 nested LOOW 中評估各 horizon。
5. 比較精度、profile cost 與方向差異。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/short_window_profile_features.csv",
    "analysis_outputs/nsys_gpu_metrics_long.csv",
    "analysis_outputs/iteration_validation.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/extract_short_window_features.py"],
    ["analysis/run_short_window_models.py"],
    ["analysis/summarize_short_window_results.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part6_short_window/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part6_short_window/predictions.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## OOF 預測與 worst cases

顯示現有 prediction table 中 APE 最大的列，並提供 actual-vs-predicted 圖。

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part6_short_window/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part6_short_window/predictions.csv"

predictions = load_csv(PREDICTIONS_PATH)
display(predictions.sort_values("ape_percent", ascending=False).head(15).reset_index(drop=True))

required = {"actual_ms", "predicted_ms", "ape_percent"}
if required.issubset(predictions.columns):
    view = predictions.copy()
    if "deployment_valid" in view.columns:
        view = view[view["deployment_valid"].astype(str).str.lower().eq("true")]
    ax = view.plot.scatter(x="actual_ms", y="predicted_ms", c="ape_percent", cmap="viridis", figsize=(6, 5))
    low = min(view["actual_ms"].min(), view["predicted_ms"].min())
    high = max(view["actual_ms"].max(), view["predicted_ms"].max())
    ax.plot([low, high], [low, high], "--", color="black", linewidth=1)
    ax.set_title("OOF prediction：actual vs predicted")
    plt.tight_layout()
    plt.show()

In [ ]:
window_metrics = load_csv(METRICS_PATH)
best_by_window = (
    window_metrics.sort_values("mape_percent")
    .groupby(["direction", "window_iterations"], as_index=False)
    .first()
)
display(best_by_window)
best_by_window.pivot(index="window_iterations", columns="direction", values="mape_percent").plot(
    marker="o", figsize=(8, 4), ylabel="MAPE (%)", title="Best model by profiling horizon"
)
plt.tight_layout()
plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part6_short_window/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 6: causal short-window profiling experiment

Each K-iteration feature row uses only Nsight samples within TRAIN_ITER_0021 through TRAIN_ITER_(0020+K). No later iteration contributes to that row.

Behavior-only models exclude GPU sample count, NVTX duration, timestamps, window span, source ground-truth runtime, and every target runtime field. Duration-assisted models are reported separately.

Outer evaluation is leave-one-workload-out. Ridge alpha is selected inside the 23-workload outer training set using shuffled 5-fold validation; scaling and variance filtering are inner-fold local.

| Direction | K | Model | Inputs | MAE ms | MAPE | Median APE | Max APE | R2 |
|---|---:|---|---:|---:|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | 5 | behavior_mean_duration_ridge | 15 | 1.266 | 3.20% | 2.15% | 11.36% | 0.9933 |
| RTX4090_to_RTX5090 | 5 | behavior_distribution_ridge | 36 | 1.630 | 3.58% | 2.87% | 11.23% | 0.9794 |
| RTX4090_to_RTX5090 | 5 | behavior_compact_cycle_ridge | 19 | 1.724 | 4.16% | 3.58% | 13.03% | 0.9871 |
| RTX4090_to_RTX5090 | 5 | behavior_mean_ridge | 14 | 2.482 | 4.72% | 2.60% | 24.82% | 0.9167 |
| RTX4090_to_RTX5090 | 5 | duration_only_ridge | 4 | 3.518 | 7.36% | 6.76% | 16.98% | 0.9280 |
| RTX4090_to_RTX5090 | 5 | behavior_cycle_ridge | 48 | 15.866 | 19.70% | 6.60% | 294.41% | -9.6496 |
| RTX4090_to_RTX5090 | 10 | behavior_mean_duration_ridge | 15 | 1.611 | 3.34% | 2.04% | 12.36% | 0.9766 |
| RTX4090_to_RTX5090 | 10 | behavior_mean_ridge | 14 | 1.952 | 3.96% | 2.64% | 14.84% | 0.9657 |
| RTX4090_to_RTX5090 | 10 | behavior_compact_cycle_ridge | 19 | 2.539 | 4.95% | 4.37% | 17.68% | 0.9499 |
| RTX4090_to_RTX5090 | 10 | duration_only_ridge | 4 | 3.391 | 7.28% | 6.34% | 18.26% | 0.9412 |
| RTX4090_to_RTX5090 | 10 | behavior_distribution_ridge | 36 | 7.396 | 8.74% | 2.33% | 143.70% | -1.5334 |
| RTX4090_to_RTX5090 | 10 | behavior_cycle_ridge | 48 | 53.054 | 53.89% | 4.89% | 1162.80% | -164.4580 |
| RTX4090_to_RTX5090 | 20 | behavior_mean_duration_ridge | 15 | 1.240 | 3.10% | 2.49% | 10.02% | 0.9941 |
| RTX4090_to_RTX5090 | 20 | behavior_mean_ridge | 14 | 2.688 | 4.58% | 2.60% | 31.90% | 0.8685 |
| RTX4090_to_RTX5090 | 20 | behavior_compact_cycle_ridge | 19 | 2.294 | 4.89% | 4.47% | 10.78% | 0.9736 |
| RTX4090_to_RTX5090 | 20 | duration_only_ridge | 4 | 3.271 | 6.98% | 5.79% | 18.89% | 0.9423 |
| RTX4090_to_RTX5090 | 20 | behavior_cycle_ridge | 48 | 6.040 | 8.06% | 3.19% | 100.00% | -0.2397 |
| RTX4090_to_RTX5090 | 20 | behavior_distribution_ridge | 36 | 11.598 | 12.87% | 2.60% | 237.87% | -5.9321 |
| RTX4090_to_RTX5090 | 50 | behavior_mean_duration_ridge | 15 | 1.226 | 3.06% | 2.25% | 10.16% | 0.9945 |
| RTX4090_to_RTX5090 | 50 | behavior_mean_ridge | 14 | 1.815 | 3.64% | 2.41% | 15.46% | 0.9648 |
| RTX4090_to_RTX5090 | 50 | behavior_distribution_ridge | 36 | 3.024 | 4.91% | 2.42% | 38.97% | 0.8044 |
| RTX4090_to_RTX5090 | 50 | behavior_compact_cycle_ridge | 19 | 2.609 | 5.16% | 3.42% | 24.35% | 0.9155 |
| RTX4090_to_RTX5090 | 50 | duration_only_ridge | 4 | 2.978 | 6.15% | 5.68% | 16.36% | 0.9462 |
| RTX4090_to_RTX5090 | 50 | behavior_cycle_ridge | 48 | 7.420 | 12.03% | 5.99% | 78.76% | 0.2225 |
| RTX4090_to_RTX5090 | 108 | behavior_mean_duration_ridge | 15 | 1.799 | 3.60% | 2.17% | 15.72% | 0.9646 |
| RTX4090_to_RTX5090 | 108 | behavior_mean_ridge | 14 | 2.447 | 4.18% | 2.26% | 31.20% | 0.8755 |
| RTX4090_to_RTX5090 | 108 | behavior_compact_cycle_ridge | 19 | 2.278 | 4.38% | 3.20% | 18.74% | 0.9474 |
| RTX4090_to_RTX5090 | 108 | behavior_distribution_ridge | 36 | 3.495 | 5.11% | 2.18% | 52.46% | 0.6555 |
| RTX4090_to_RTX5090 | 108 | duration_only_ridge | 4 | 3.006 | 6.27% | 4.92% | 16.94% | 0.9460 |
| RTX4090_to_RTX5090 | 108 | behavior_cycle_ridge | 48 | 3.700 | 6.72% | 5.95% | 38.71% | 0.7989 |
| RTX5090_to_RTX4090 | 5 | behavior_distribution_ridge | 36 | 4.406 | 7.18% | 5.47% | 20.21% | 0.9087 |
| RTX5090_to_RTX4090 | 5 | behavior_mean_duration_ridge | 15 | 3.474 | 7.39% | 5.49% | 48.72% | 0.9500 |
| RTX5090_to_RTX4090 | 5 | behavior_mean_ridge | 14 | 4.956 | 8.26% | 5.66% | 26.16% | 0.8847 |
| RTX5090_to_RTX4090 | 5 | behavior_compact_cycle_ridge | 19 | 5.031 | 8.50% | 6.89% | 24.15% | 0.8911 |
| RTX5090_to_RTX4090 | 5 | behavior_cycle_ridge | 48 | 4.992 | 8.92% | 9.61% | 19.98% | 0.9174 |
| RTX5090_to_RTX4090 | 5 | duration_only_ridge | 4 | 7.438 | 11.24% | 10.71% | 61.70% | 0.4926 |
| RTX5090_to_RTX4090 | 10 | behavior_mean_duration_ridge | 15 | 2.820 | 6.17% | 4.81% | 33.32% | 0.9752 |
| RTX5090_to_RTX4090 | 10 | behavior_mean_ridge | 14 | 3.872 | 6.76% | 5.70% | 20.51% | 0.9423 |
| RTX5090_to_RTX4090 | 10 | behavior_distribution_ridge | 36 | 4.467 | 7.04% | 5.65% | 20.53% | 0.9071 |
| RTX5090_to_RTX4090 | 10 | behavior_compact_cycle_ridge | 19 | 4.836 | 8.13% | 6.43% | 23.81% | 0.8984 |
| RTX5090_to_RTX4090 | 10 | behavior_cycle_ridge | 48 | 4.891 | 8.15% | 8.00% | 23.28% | 0.9038 |
| RTX5090_to_RTX4090 | 10 | duration_only_ridge | 4 | 5.463 | 9.17% | 9.50% | 35.57% | 0.8114 |
| RTX5090_to_RTX4090 | 20 | behavior_mean_duration_ridge | 15 | 2.657 | 5.17% | 5.16% | 10.81% | 0.9776 |
| RTX5090_to_RTX4090 | 20 | behavior_mean_ridge | 14 | 3.878 | 6.55% | 4.67% | 16.50% | 0.9344 |
| RTX5090_to_RTX4090 | 20 | behavior_distribution_ridge | 36 | 4.038 | 6.65% | 5.72% | 16.62% | 0.9331 |
| RTX5090_to_RTX4090 | 20 | duration_only_ridge | 4 | 3.760 | 7.66% | 7.75% | 16.21% | 0.9578 |
| RTX5090_to_RTX4090 | 20 | behavior_compact_cycle_ridge | 19 | 4.726 | 8.12% | 7.07% | 23.26% | 0.9069 |
| RTX5090_to_RTX4090 | 20 | behavior_cycle_ridge | 48 | 4.865 | 8.44% | 7.89% | 21.46% | 0.9132 |
| RTX5090_to_RTX4090 | 50 | behavior_mean_duration_ridge | 15 | 2.092 | 4.49% | 3.58% | 10.61% | 0.9887 |
| RTX5090_to_RTX4090 | 50 | behavior_cycle_ridge | 48 | 3.852 | 6.55% | 5.12% | 19.45% | 0.9405 |
| RTX5090_to_RTX4090 | 50 | behavior_mean_ridge | 14 | 3.792 | 6.60% | 6.52% | 16.35% | 0.9434 |
| RTX5090_to_RTX4090 | 50 | behavior_distribution_ridge | 36 | 4.544 | 7.15% | 5.18% | 18.66% | 0.9030 |
| RTX5090_to_RTX4090 | 50 | behavior_compact_cycle_ridge | 19 | 4.316 | 7.58% | 6.76% | 18.76% | 0.9237 |
| RTX5090_to_RTX4090 | 50 | duration_only_ridge | 4 | 4.366 | 7.96% | 8.10% | 22.64% | 0.9067 |
| RTX5090_to_RTX4090 | 108 | behavior_mean_duration_ridge | 15 | 1.965 | 4.32% | 3.54% | 10.23% | 0.9907 |
| RTX5090_to_RTX4090 | 108 | behavior_distribution_ridge | 36 | 3.577 | 5.82% | 5.02% | 14.86% | 0.9440 |
| RTX5090_to_RTX4090 | 108 | behavior_compact_cycle_ridge | 19 | 3.681 | 6.10% | 5.83% | 20.66% | 0.9391 |
| RTX5090_to_RTX4090 | 108 | behavior_cycle_ridge | 48 | 3.342 | 6.30% | 5.52% | 18.59% | 0.9573 |
| RTX5090_to_RTX4090 | 108 | behavior_mean_ridge | 14 | 4.147 | 6.91% | 6.14% | 19.00% | 0.9174 |
| RTX5090_to_RTX4090 | 108 | duration_only_ridge | 4 | 3.524 | 7.28% | 6.03% | 16.47% | 0.9630 |
| both_directions_combined | 5 | behavior_mean_duration_ridge | 15 | 2.370 | 5.30% | 3.03% | 48.72% | 0.9673 |
| both_directions_combined | 5 | behavior_distribution_ridge | 36 | 3.018 | 5.38% | 3.62% | 20.21% | 0.9371 |
| both_directions_combined | 5 | behavior_compact_cycle_ridge | 19 | 3.378 | 6.33% | 4.64% | 24.15% | 0.9294 |
| both_directions_combined | 5 | behavior_mean_ridge | 14 | 3.719 | 6.49% | 3.68% | 26.16% | 0.8988 |
| both_directions_combined | 5 | duration_only_ridge | 4 | 5.478 | 9.30% | 9.25% | 61.70% | 0.6666 |
| both_directions_combined | 5 | behavior_cycle_ridge | 48 | 10.429 | 14.31% | 8.02% | 294.41% | -3.0880 |
| both_directions_combined | 10 | behavior_mean_duration_ridge | 15 | 2.216 | 4.76% | 3.64% | 33.32% | 0.9762 |
| both_directions_combined | 10 | behavior_mean_ridge | 14 | 2.912 | 5.36% | 3.48% | 20.51% | 0.9522 |
| both_directions_combined | 10 | behavior_compact_cycle_ridge | 19 | 3.688 | 6.54% | 4.67% | 23.81% | 0.9197 |
| both_directions_combined | 10 | behavior_distribution_ridge | 36 | 5.931 | 7.89% | 3.62% | 143.70% | -0.0167 |
| both_directions_combined | 10 | duration_only_ridge | 4 | 4.427 | 8.23% | 7.49% | 35.57% | 0.8639 |
| both_directions_combined | 10 | behavior_cycle_ridge | 48 | 28.973 | 31.02% | 6.36% | 1162.80% | -61.7984 |
| both_directions_combined | 20 | behavior_mean_duration_ridge | 15 | 1.948 | 4.14% | 2.84% | 10.81% | 0.9842 |
| both_directions_combined | 20 | behavior_mean_ridge | 14 | 3.283 | 5.57% | 3.22% | 31.90% | 0.9105 |
| both_directions_combined | 20 | behavior_compact_cycle_ridge | 19 | 3.510 | 6.50% | 5.35% | 23.26% | 0.9338 |
| both_directions_combined | 20 | duration_only_ridge | 4 | 3.516 | 7.32% | 6.41% | 18.89% | 0.9527 |
| both_directions_combined | 20 | behavior_cycle_ridge | 48 | 5.452 | 8.25% | 6.34% | 100.00% | 0.4776 |
| both_directions_combined | 20 | behavior_distribution_ridge | 36 | 7.818 | 9.76% | 3.90% | 237.87% | -1.6690 |
| both_directions_combined | 50 | behavior_mean_duration_ridge | 15 | 1.659 | 3.77% | 2.73% | 10.61% | 0.9911 |
| both_directions_combined | 50 | behavior_mean_ridge | 14 | 2.804 | 5.12% | 3.24% | 16.35% | 0.9525 |
| both_directions_combined | 50 | behavior_distribution_ridge | 36 | 3.784 | 6.03% | 4.36% | 38.97% | 0.8673 |
| both_directions_combined | 50 | behavior_compact_cycle_ridge | 19 | 3.463 | 6.37% | 4.77% | 24.35% | 0.9219 |
| both_directions_combined | 50 | duration_only_ridge | 4 | 3.672 | 7.06% | 6.43% | 22.64% | 0.9233 |
| both_directions_combined | 50 | behavior_cycle_ridge | 48 | 5.636 | 9.29% | 5.75% | 78.76% | 0.6693 |
| both_directions_combined | 108 | behavior_mean_duration_ridge | 15 | 1.882 | 3.96% | 2.61% | 15.72% | 0.9809 |
| both_directions_combined | 108 | behavior_compact_cycle_ridge | 19 | 2.980 | 5.24% | 4.07% | 20.66% | 0.9433 |
| both_directions_combined | 108 | behavior_distribution_ridge | 36 | 3.536 | 5.46% | 3.69% | 52.46% | 0.8356 |
| both_directions_combined | 108 | behavior_mean_ridge | 14 | 3.297 | 5.55% | 2.74% | 31.20% | 0.9030 |
| both_directions_combined | 108 | behavior_cycle_ridge | 48 | 3.521 | 6.51% | 5.95% | 38.71% | 0.8980 |
| both_directions_combined | 108 | duration_only_ridge | 4 | 3.265 | 6.78% | 5.34% | 16.94% | 0.9572 |

## Best strict behavior-only model by window

| Direction | K | Model | MAPE |
|---|---:|---|---:|
| RTX4090_to_RTX5090 | 5 | behavior_distribution_ridge | 3.58% |
| RTX4090_to_RTX5090 | 10 | behavior_mean_ridge | 3.96% |
| RTX4090_to_RTX5090 | 20 | behavior_mean_ridge | 4.58% |
| RTX4090_to_RTX5090 | 50 | behavior_mean_ridge | 3.64% |
| RTX4090_to_RTX5090 | 108 | behavior_mean_ridge | 4.18% |
| RTX5090_to_RTX4090 | 5 | behavior_distribution_ridge | 7.18% |
| RTX5090_to_RTX4090 | 10 | behavior_mean_ridge | 6.76% |
| RTX5090_to_RTX4090 | 20 | behavior_mean_ridge | 6.55% |
| RTX5090_to_RTX4090 | 50 | behavior_cycle_ridge | 6.55% |
| RTX5090_to_RTX4090 | 108 | behavior_distribution_ridge | 5.82% |

## Hyperparameter selection frequency

- RTX4090_to_RTX5090 / K=5 / behavior_compact_cycle_ridge: `{"model__alpha": 1.0}`: 16; `{"model__alpha": 0.1}`: 5; `{"model__alpha": 0.01}`: 2; `{"model__alpha": 10.0}`: 1
- RTX4090_to_RTX5090 / K=5 / behavior_cycle_ridge: `{"model__alpha": 100.0}`: 13; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 0.1}`: 3; `{"model__alpha": 10.0}`: 1
- RTX4090_to_RTX5090 / K=5 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 9; `{"model__alpha": 0.01}`: 8; `{"model__alpha": 0.1}`: 7
- RTX4090_to_RTX5090 / K=5 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 20; `{"model__alpha": 0.01}`: 3; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=5 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 17; `{"model__alpha": 0.01}`: 6; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=5 / duration_only_ridge: `{"model__alpha": 1.0}`: 21; `{"model__alpha": 0.1}`: 2; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / K=10 / behavior_compact_cycle_ridge: `{"model__alpha": 1.0}`: 14; `{"model__alpha": 0.1}`: 8; `{"model__alpha": 0.01}`: 1; `{"model__alpha": 10.0}`: 1
- RTX4090_to_RTX5090 / K=10 / behavior_cycle_ridge: `{"model__alpha": 100.0}`: 10; `{"model__alpha": 10.0}`: 10; `{"model__alpha": 0.01}`: 2; `{"model__alpha": 0.1}`: 2
- RTX4090_to_RTX5090 / K=10 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 12; `{"model__alpha": 0.01}`: 9; `{"model__alpha": 0.1}`: 3
- RTX4090_to_RTX5090 / K=10 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 17; `{"model__alpha": 0.01}`: 5; `{"model__alpha": 1.0}`: 2
- RTX4090_to_RTX5090 / K=10 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 16; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=10 / duration_only_ridge: `{"model__alpha": 1.0}`: 21; `{"model__alpha": 0.1}`: 2; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / K=20 / behavior_compact_cycle_ridge: `{"model__alpha": 0.1}`: 13; `{"model__alpha": 1.0}`: 10; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / K=20 / behavior_cycle_ridge: `{"model__alpha": 0.01}`: 17; `{"model__alpha": 0.1}`: 3; `{"model__alpha": 10.0}`: 2; `{"model__alpha": 1.0}`: 2
- RTX4090_to_RTX5090 / K=20 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 15; `{"model__alpha": 0.1}`: 6; `{"model__alpha": 0.01}`: 3
- RTX4090_to_RTX5090 / K=20 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 15; `{"model__alpha": 0.01}`: 8; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=20 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 17; `{"model__alpha": 0.01}`: 6; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=20 / duration_only_ridge: `{"model__alpha": 1.0}`: 18; `{"model__alpha": 0.1}`: 3; `{"model__alpha": 0.01}`: 3
- RTX4090_to_RTX5090 / K=50 / behavior_compact_cycle_ridge: `{"model__alpha": 0.1}`: 15; `{"model__alpha": 1.0}`: 8; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / K=50 / behavior_cycle_ridge: `{"model__alpha": 10.0}`: 10; `{"model__alpha": 1.0}`: 8; `{"model__alpha": 0.01}`: 5; `{"model__alpha": 0.1}`: 1
- RTX4090_to_RTX5090 / K=50 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 15; `{"model__alpha": 0.1}`: 8; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / K=50 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 20; `{"model__alpha": 0.01}`: 4
- RTX4090_to_RTX5090 / K=50 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 20; `{"model__alpha": 0.01}`: 4
- RTX4090_to_RTX5090 / K=50 / duration_only_ridge: `{"model__alpha": 1.0}`: 15; `{"model__alpha": 0.1}`: 5; `{"model__alpha": 0.01}`: 4
- RTX4090_to_RTX5090 / K=108 / behavior_compact_cycle_ridge: `{"model__alpha": 0.1}`: 14; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 0.01}`: 3
- RTX4090_to_RTX5090 / K=108 / behavior_cycle_ridge: `{"model__alpha": 1.0}`: 12; `{"model__alpha": 0.1}`: 7; `{"model__alpha": 0.01}`: 4; `{"model__alpha": 10.0}`: 1
- RTX4090_to_RTX5090 / K=108 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 10; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 0.1}`: 6; `{"model__alpha": 10.0}`: 1
- RTX4090_to_RTX5090 / K=108 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 16; `{"model__alpha": 0.01}`: 5; `{"model__alpha": 1.0}`: 3
- RTX4090_to_RTX5090 / K=108 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 19; `{"model__alpha": 0.01}`: 4; `{"model__alpha": 1.0}`: 1
- RTX4090_to_RTX5090 / K=108 / duration_only_ridge: `{"model__alpha": 1.0}`: 15; `{"model__alpha": 0.1}`: 6; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / K=5 / behavior_compact_cycle_ridge: `{"model__alpha": 1.0}`: 12; `{"model__alpha": 10.0}`: 12
- RTX5090_to_RTX4090 / K=5 / behavior_cycle_ridge: `{"model__alpha": 1.0}`: 17; `{"model__alpha": 10.0}`: 5; `{"model__alpha": 0.1}`: 2
- RTX5090_to_RTX4090 / K=5 / behavior_distribution_ridge: `{"model__alpha": 0.1}`: 8; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 10.0}`: 2
- RTX5090_to_RTX4090 / K=5 / behavior_mean_duration_ridge: `{"model__alpha": 1.0}`: 11; `{"model__alpha": 0.1}`: 9; `{"model__alpha": 0.01}`: 3; `{"model__alpha": 10.0}`: 1
- RTX5090_to_RTX4090 / K=5 / behavior_mean_ridge: `{"model__alpha": 1.0}`: 15; `{"model__alpha": 0.1}`: 6; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / K=5 / duration_only_ridge: `{"model__alpha": 1.0}`: 17; `{"model__alpha": 0.1}`: 6; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / K=10 / behavior_compact_cycle_ridge: `{"model__alpha": 10.0}`: 14; `{"model__alpha": 1.0}`: 6; `{"model__alpha": 0.1}`: 3; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / K=10 / behavior_cycle_ridge: `{"model__alpha": 10.0}`: 13; `{"model__alpha": 1.0}`: 11
- RTX5090_to_RTX4090 / K=10 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 17; `{"model__alpha": 0.1}`: 5; `{"model__alpha": 10.0}`: 2
- RTX5090_to_RTX4090 / K=10 / behavior_mean_duration_ridge: `{"model__alpha": 1.0}`: 10; `{"model__alpha": 0.1}`: 10; `{"model__alpha": 0.01}`: 3; `{"model__alpha": 10.0}`: 1
- RTX5090_to_RTX4090 / K=10 / behavior_mean_ridge: `{"model__alpha": 1.0}`: 12; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 0.1}`: 5
- RTX5090_to_RTX4090 / K=10 / duration_only_ridge: `{"model__alpha": 1.0}`: 22; `{"model__alpha": 0.1}`: 2
- RTX5090_to_RTX4090 / K=20 / behavior_compact_cycle_ridge: `{"model__alpha": 10.0}`: 13; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 0.1}`: 4
- RTX5090_to_RTX4090 / K=20 / behavior_cycle_ridge: `{"model__alpha": 10.0}`: 16; `{"model__alpha": 1.0}`: 8
- RTX5090_to_RTX4090 / K=20 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 19; `{"model__alpha": 10.0}`: 4; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / K=20 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 12; `{"model__alpha": 1.0}`: 9; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / K=20 / behavior_mean_ridge: `{"model__alpha": 0.1}`: 10; `{"model__alpha": 1.0}`: 8; `{"model__alpha": 0.01}`: 6
- RTX5090_to_RTX4090 / K=20 / duration_only_ridge: `{"model__alpha": 1.0}`: 23; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / K=50 / behavior_compact_cycle_ridge: `{"model__alpha": 1.0}`: 20; `{"model__alpha": 10.0}`: 3; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / K=50 / behavior_cycle_ridge: `{"model__alpha": 1.0}`: 18; `{"model__alpha": 10.0}`: 5; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / K=50 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 17; `{"model__alpha": 10.0}`: 5; `{"model__alpha": 0.1}`: 2
- RTX5090_to_RTX4090 / K=50 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 14; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / K=50 / behavior_mean_ridge: `{"model__alpha": 0.01}`: 11; `{"model__alpha": 1.0}`: 11; `{"model__alpha": 0.1}`: 2
- RTX5090_to_RTX4090 / K=50 / duration_only_ridge: `{"model__alpha": 1.0}`: 19; `{"model__alpha": 0.1}`: 5
- RTX5090_to_RTX4090 / K=108 / behavior_compact_cycle_ridge: `{"model__alpha": 10.0}`: 12; `{"model__alpha": 1.0}`: 12
- RTX5090_to_RTX4090 / K=108 / behavior_cycle_ridge: `{"model__alpha": 10.0}`: 17; `{"model__alpha": 1.0}`: 7
- RTX5090_to_RTX4090 / K=108 / behavior_distribution_ridge: `{"model__alpha": 1.0}`: 20; `{"model__alpha": 10.0}`: 3; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / K=108 / behavior_mean_duration_ridge: `{"model__alpha": 0.1}`: 18; `{"model__alpha": 1.0}`: 5; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / K=108 / behavior_mean_ridge: `{"model__alpha": 0.01}`: 12; `{"model__alpha": 1.0}`: 8; `{"model__alpha": 0.1}`: 4
- RTX5090_to_RTX4090 / K=108 / duration_only_ridge: `{"model__alpha": 1.0}`: 18; `{"model__alpha": 0.1}`: 6

## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。